In [ ]:
import torch
from datetime import date, timedelta
import numpy as np
from netCDF4 import Dataset
from MyDataPreparation import CustomDataset
from torch.utils.data import DataLoader
from autoencoder import Encoder, Decoder

In [ ]:
dates = {
    '2010': {'start': '0711', 'finish': '1009'},
    '2011': {'start': '0625', 'finish': '1029'},
    '2012': {'start': '0701', 'finish': '1105'},
    '2013': {'start': '0713', 'finish': '1018'},
    '2014': {'start': '0716', 'finish': '1023'},
    '2015': {'start': '0630', 'finish': '1025'},
    '2016': {'start': '0709', 'finish': '1031'},
    '2017': {'start': '0715', 'finish': '1020'},
    '2018': {'start': '0801', 'finish': '1031'},
    '2019': {'start': '0710', 'finish': '1025'},
    '2020': {'start': '0707', 'finish': '1031'},
    '2021': {'start': '0710', 'finish': '1025'},
    '2022': {'start': '0701', 'finish': '1031'},
    '2023': {'start': '0720', 'finish': '1010'},
}

In [ ]:
dataset = CustomDataset(dates_dict=dates, borders=[80,70,55,105])

In [ ]:
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

In [ ]:
data, mask = next(iter(dataloader))

In [ ]:
data.shape, mask.shape

In [ ]:
encoder = Encoder(in_channels=1, H=74, W=52, expansions=[4, 4, 4, 4, 4], n_blocks=32, decreases=[2, 2, 2, 2, 2], bottleneck=64)
decoder = Decoder(in_features=encoder.bottleneck, start_channels=1024, finish_channels=encoder.in_channels, n_layers=5,
                    expansion_value=0.25, increase_value=2, H=3, W=2, H_out=74, W_out=52)

In [ ]:
encoder.cuda();
decoder.cuda();

In [ ]:
loss_function=torch.nn.MSELoss()

In [ ]:
data_gpu = data.to(device='cuda', dtype=torch.float)
mask_gpu = mask.to(device='cuda', dtype=torch.float)

encoded_data = encoder.forward(data_gpu)
decoded_data = decoder.forward(encoded_data)

data_gpu_masked = data_gpu[mask_gpu == 1]
result_masked = decoded_data[mask_gpu == 1]

loss = loss_function(data_gpu_masked, result_masked)

In [ ]:
loss

In [ ]:
mask.sum()